# Zero Tic‑Tac‑Toe — CCP (Hotz–Miller) demo (revised)This notebook uses the revised, fast CCP implementation with memoized continuation values.

In [1]:
import sys, json
import numpy as np
# from my_testing.tictactoe.NRLS.tictactoe_structural_full_demo import theta_hat_newton

sys.path.append('/data')
import zero_ttt_core as Z
import ccp_zero_ttt as CCP
import zero_ttt_ccp_utils as U
import ccp_io as IO

print('Loaded:', Z.__name__, CCP.__name__)

Loaded: zero_ttt_core ccp_zero_ttt


## 1) Generate dataset

In [2]:
theta_gen = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0])
data = CCP.generate_dataset(n_games=50, policy='theta', theta=theta_gen, depth=3, seed=0)  # small & fastlen(data)

In [3]:
IO.save_dataset_jsonl("zttt_dataset.jsonl.gz", data)   # one JSON per line

'zttt_dataset.jsonl.gz'

In [ ]:
# Later:
# data2 = IO.load_dataset_jsonl("zttt_dataset.jsonl.gz")

## 2) Estimate empirical CCPs

In [4]:
ccp = CCP.estimate_ccp(data, alpha=0.5)
len(ccp)

7

## 3) Build continuation offsets (memoized)

In [5]:
beta = 1.0
H = 3
offsets = CCP.build_offsets(data, ccp, H=H, beta=beta)
len(offsets)
print(offsets)

[array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]), array([-1.00000000e+09,  0.00000000e+00,  0.00000000e+00, -9.04198317e-02,
       -8.65765756e-02, -8.17563389e-02, -9.04198317e-02, -8.65765756e-02,
       -8.17563389e-02, -9.04198317e-02, -8.65765756e-02, -8.17563389e-02,
       -9.04198317e-02, -8.65765756e-02, -8.17563389e-02, -9.79699994e-02,
       -1.09237472e-01, -1.22634508e-01, -9.04198317e-02, -8.65765756e-02,
       -8.17563389e-02, -9.79699994e-02, -1.09237472e-01, -1.22634508e-01,
       -9.04198317e-02, -8.65765756e-02, -8.17563389e-02]), array([-1.00000000e+09, -1.00000000e+09,  0.00000000e+00, -8.99594036e-02,
       -9.09360550e-02, -8.58575009e-02, -8.99594036e-02, -9.09360550e-02,
       -8.58575009e-02, -8.99594036e-02, -9.09360550e-02, -8.58575009e-02,
       -8.99594036e-02, -9.09360550e-02, -8.58575009e-02, -1.00367848e-01,
       -1.14583869e-01, -1.28786251e-01, -8.99594036e-02, -9.0936

In [7]:
arr = np.array(offsets)
np.save("offsets_array.npy", arr)

In [ ]:
offsets = np.load("offsets_array.npy")
len(offsets)

## 4) Fit θ by masked multinomial logit with offsets

In [6]:
theta_hat = CCP.fit_theta(data, offsets, beta=beta, lr=0.1, iters=1000)
len(theta_hat)

iter  100  avg ll=-2.0818
iter  200  avg ll=-1.7935
iter  300  avg ll=-1.6704
iter  400  avg ll=-1.6024
iter  500  avg ll=-1.5591
iter  600  avg ll=-1.5292
iter  700  avg ll=-1.5074
iter  800  avg ll=-1.4909
iter  900  avg ll=-1.4781
iter 1000  avg ll=-1.4678


14

## 5) Held-out log-likelihood (80/20 split)

In [6]:
# train, test = CCP.train_test_split(data, test_ratio=0.2, seed=1)
# off_tr = CCP.build_offsets(train, ccp, H=H, beta=beta)
# off_te = CCP.build_offsets(test,  ccp, H=H, beta=beta)
# theta_hat2 = CCP.fit_theta(train, off_tr, beta=beta, lr=0.1, iters=200)
# ll_tr = CCP.loglik(train, off_tr, theta_hat2, beta=beta)
# ll_te = CCP.loglik(test,  off_te, theta_hat2, beta=beta)
# print('avg ll train:', ll_tr/len(train))
# print('avg ll test :', ll_te/len(test))

iter   50  avg ll=-2.5596
iter  100  avg ll=-2.3330
iter  150  avg ll=-2.2112
iter  200  avg ll=-2.1452
avg ll train: -2.1442462082048057
avg ll test : -2.081317203690327


## 6) Inspect policy on a sample state

In [8]:
s = data[0].state
probs = CCP.policy_probs(s, theta_hat, ccp, H=H, beta=beta)
top = sorted([(float(probs[a]), a) for a in range(27)], reverse=True)[:10]
def fmt(aid):
    v,i = CCP.id_to_action(aid)
    return f'(v={v}, i={i})'
[(p, fmt(a)) for p,a in top]

[(0.053744840615547, '(v=1, i=8)'),
 (0.053744840615547, '(v=1, i=7)'),
 (0.053744840615547, '(v=1, i=6)'),
 (0.053744840615547, '(v=1, i=5)'),
 (0.053744840615547, '(v=1, i=4)'),
 (0.053744840615547, '(v=1, i=3)'),
 (0.053744840615547, '(v=1, i=2)'),
 (0.053744840615547, '(v=1, i=1)'),
 (0.053744840615547, '(v=1, i=0)'),
 (0.037234150697344925, '(v=2, i=8)')]

## 7) Save θ for the website UI

In [9]:
theta_path = 'theta_ccp.json'
with open(theta_path, 'w') as f:
    json.dump(theta_hat.tolist(), f)

## 8) Play Test

In [10]:
# 3a) Bot vs Bot
winner, history, final_state = U.play_game(theta_X=theta_hat, theta_O=theta_hat, ccp=ccp,
                                           H=3, beta=1.0, mode_X="argmax", mode_O="argmax",
                                           verbose=False)
print("Winner:", winner)  # 1=X, 2=O, 0=draw


Winner: 0


In [11]:
# 3b) Human vs Bot (run in a terminal/interactive cell)
U.human_vs_bot(ccp=ccp, theta_bot=theta_hat, human="X", H=3, beta=1.0, mode_bot="argmax")

Welcome to Zero TTT (CCP bot). You are X
Cells are indexed 0..8:
0 1 2
3 4 5
6 7 8
 ·  |  ·  |  · 
---------------
 ·  |  ·  |  · 
---------------
 ·  |  ·  |  · 
X inv: 1×2  2×2  3×2     O inv: 1×2  2×2  3×2     to_move: X
 ·  |  ·  |  · 
---------------
 ·  | X3  |  · 
---------------
 ·  |  ·  |  · 
X inv: 1×2  2×2  3×1     O inv: 1×2  2×2  3×2     to_move: O
Bot plays: v=2 at i=1
 ·  | O2  |  · 
---------------
 ·  | X3  |  · 
---------------
 ·  |  ·  |  · 
X inv: 1×2  2×2  3×1     O inv: 1×2  2×1  3×2     to_move: X
 ·  | O2  |  · 
---------------
 ·  | X3  |  · 
---------------
X1  |  ·  |  · 
X inv: 1×1  2×2  3×1     O inv: 1×2  2×1  3×2     to_move: O
Bot plays: v=2 at i=6
 ·  | O2  |  · 
---------------
 ·  | X3  |  · 
---------------
O2  |  ·  |  · 
X inv: 1×1  2×2  3×1     O inv: 1×2  2×0  3×2     to_move: X
 ·  | O2  |  · 
---------------
 ·  | X3  |  · 
---------------
O2  |  ·  | X1 
X inv: 1×0  2×2  3×1     O inv: 1×2  2×0  3×2     to_move: O
Bot plays: v=3 at i=8
 ·  |

1